# Benchmark - legacy vs optimized vs aggressive (3 layouts) Warp kernels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Forgotten/vlasov-poisson-warp/blob/main/notebooks/03_Benchmark_Optimized_vs_Legacy.ipynb)

Compares the code paths exposed by `WarpVlasovPoissonSolver`:

* `optimized=False`                  - 1:1 port of the JAX semi-Lagrangian scheme.
* `optimized=True`                   - fused `E + H` v-step kernel and on-device
  cotangent pipeline for the adjoint, with an optional
  `record_history=False` mode that skips per-step `(nx, nv)`
  device-to-host copies.
* `optimized=True, aggressive=True` - additionally enables F5 (fused
  semilag-x + rho reduction) and M3 (Strang-merge across time
  steps).  Three layouts are exposed:

  * `aggressive_layout='cpu_fused'` - F5 row-sweep at `dim=nx`.
    Best on CPU; collapses GPU parallelism.
  * `aggressive_layout='gpu_safe'`  - M3 only, no F5.  Uses
    `semilag_x_full_kernel` at `dim=(nx, nv)` and a separate
    `compute_rho_kernel` so threads stay coalesced and the GPU
    keeps full occupancy.
  * `aggressive_layout='tiled'`     - F5 + M3 with
    block-cooperative reduction at `dim=(nx, num_tiles)`,
    `num_tiles = ceil(nv / aggressive_tile_size)`.  Compromise
    between cpu_fused and gpu_safe.

  Forward only - `jax.grad` raises on aggressive solvers.
  Linear semi-Lagrangian is non-commutative, so all aggressive
  layouts produce a slightly *less diffusive* solution than the
  unmerged Strang split; expect O(dx^2 * dt) drift away from
  the optimized path's `f_final`.

We measure forward-only wall time at several grid sizes for
every aggressive layout (with and without `record_history`),
plus the legacy / optimized baselines and `jax.grad` (legacy
and optimized only).  Optimized is bit-for-bit equivalent to
legacy on every output (`tests/test_optimized.py`);
aggressive layouts agree with optimized to a small tolerance
and conserve mass / preserve equilibrium
(`tests/test_aggressive.py`).

**Expected behavior**: the gap is modest on CPU - most of what
the optimized path eliminates is per-step kernel-launch and
host/device transfer overhead, which is cheap on a single CPU
process.  On a CUDA build of Warp the per-step transfer
elimination across all paths is much more visible (especially
on the gradient path).  The aggressive layouts trade
differently on each platform: `cpu_fused` is fastest on CPU,
`gpu_safe` keeps GPU occupancy at any grid size, and `tiled`
tries to balance both.


## Setup


In [ ]:
# Colab setup - skip if running locally with the package already installed.
import importlib, subprocess, sys

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg,
        ])

for pkg, pip_name in [
    ('warp', 'warp-lang'),
    ('jax', 'jax'),
    ('optax', 'optax'),
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
]:
    _ensure(pkg, pip_name)

# Install the warp_vp_solver package itself.
try:
    import warp_vp_solver  # noqa: F401
except ImportError:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/Forgotten/warp_vp_solver.git@main',
    ])

import warp as wp
wp.init()
print(f'Warp device: {wp.get_preferred_device()}')


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from warp_vp_solver import make_mesh, WarpVlasovPoissonSolver

jax.config.update('jax_enable_x64', True)


## Two-Stream problem at a chosen grid size


In [ ]:
def make_two_stream(nx, nv, *, length_x=10*np.pi, length_v=6.0):
    mesh = make_mesh(length_x, length_v, nx, nv)
    mu = 2.4
    f_eq = (
        np.exp(-0.5*(mesh.V - mu)**2)
        + np.exp(-0.5*(mesh.V + mu)**2)
    ) / (2*np.sqrt(2*np.pi))
    f_iv = (1 + 1e-3*np.cos(0.2*mesh.X)) * f_eq
    return mesh, f_eq, f_iv


## Correctness sanity check


Before timing anything, confirm the two paths really do agree on
the grid we are about to benchmark.  The full test suite checks
this much more thoroughly; this is a one-off guard.


In [ ]:
mesh, f_eq, f_iv = make_two_stream(nx=64, nv=64)
dt = 0.05; t_final = 10*dt
H = np.zeros(mesh.nx)

s_legacy = WarpVlasovPoissonSolver(
    mesh=mesh, dt=dt, f_eq=f_eq, optimized=False,
)
s_fast = WarpVlasovPoissonSolver(
    mesh=mesh, dt=dt, f_eq=f_eq, optimized=True,
)

out_l = s_legacy.run_forward(f_iv, H, t_final)
out_f = s_fast.run_forward(f_iv, H, t_final)
for name, a, b in zip(
    ('f_final', 'f_hist', 'E_hist', 'ee_hist'), out_l, out_f,
):
    np.testing.assert_array_equal(a, b)
print('legacy and optimized paths produce identical outputs.')


## Benchmark helpers


In [ ]:
def bench(fn, warmups=2, repeats=5):
    for _ in range(warmups):
        fn()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times), float(np.mean(times))

AGG_LAYOUTS = ('cpu_fused', 'gpu_safe', 'tiled')

def make_runners(nx, nv, *, dt=0.05, t_steps=20, tile_size=32):
    mesh, f_eq, f_iv = make_two_stream(nx, nv)
    t_final = t_steps * dt
    H = np.zeros(mesh.nx)
    f_iv_j = jnp.asarray(f_iv); H_j = jnp.asarray(H)
    s_legacy = WarpVlasovPoissonSolver(
        mesh=mesh, dt=dt, f_eq=f_eq, optimized=False,
    )
    s_fast = WarpVlasovPoissonSolver(
        mesh=mesh, dt=dt, f_eq=f_eq, optimized=True,
    )
    agg_solvers = {
        layout: WarpVlasovPoissonSolver(
            mesh=mesh, dt=dt, f_eq=f_eq,
            optimized=True, aggressive=True,
            aggressive_layout=layout,
            aggressive_tile_size=tile_size,
        )
        for layout in AGG_LAYOUTS
    }

    def run_legacy_full():
        s_legacy.run_forward(f_iv, H, t_final)
    def run_fast_full():
        s_fast.run_forward(f_iv, H, t_final, record_history=True)
    def run_fast_lite():
        s_fast.run_forward(f_iv, H, t_final, record_history=False)

    def make_agg_runner(solver, *, lite):
        def run():
            solver.run_forward(
                f_iv, H, t_final, record_history=not lite,
            )
        return run

    def grad_cost_factory(solver):
        def cost(H):
            f_final, _, _, ee = solver.run_forward_jax(
                f_iv_j, H, t_final,
            )
            return jnp.sum(f_final**2) + jnp.sum(ee)
        g = jax.grad(cost)
        def run():
            out = g(H_j)
            jax.block_until_ready(out)
        return run

    runners = {
        'forward legacy':                run_legacy_full,
        'forward optimized':             run_fast_full,
        'forward optimized (no fhist)':  run_fast_lite,
    }
    for layout in AGG_LAYOUTS:
        runners[f'forward aggressive [{layout}]'] = make_agg_runner(
            agg_solvers[layout], lite=False,
        )
        runners[f'forward aggressive [{layout}] (no fhist)'] = make_agg_runner(
            agg_solvers[layout], lite=True,
        )
    runners['grad legacy']    = grad_cost_factory(s_legacy)
    runners['grad optimized'] = grad_cost_factory(s_fast)
    return runners


## Run the benchmark across several grid sizes


In [ ]:
grid_sizes = [(32, 32), (64, 64), (128, 128), (256, 256)]
labels = [
    'forward legacy',
    'forward optimized',
    'forward optimized (no fhist)',
    'forward aggressive [cpu_fused]',
    'forward aggressive [cpu_fused] (no fhist)',
    'forward aggressive [gpu_safe]',
    'forward aggressive [gpu_safe] (no fhist)',
    'forward aggressive [tiled]',
    'forward aggressive [tiled] (no fhist)',
    'grad legacy',
    'grad optimized',
]
results = {label: [] for label in labels}

for nx, nv in grid_sizes:
    print(f'=== grid {nx} x {nv} ===')
    runners = make_runners(nx, nv)
    for label in labels:
        mn, mean = bench(runners[label])
        results[label].append(mn * 1e3)
        print(f'  {label:34s}  min={mn*1e3:7.2f} ms   mean={mean*1e3:7.2f} ms')


## Plot - forward-only timing


In [ ]:
x = np.arange(len(grid_sizes))
labels_x = [f'{nx}x{nv}' for nx, nv in grid_sizes]
fwd_labels = [
    'forward legacy',
    'forward optimized',
    'forward optimized (no fhist)',
    'forward aggressive [cpu_fused]',
    'forward aggressive [cpu_fused] (no fhist)',
    'forward aggressive [gpu_safe]',
    'forward aggressive [gpu_safe] (no fhist)',
    'forward aggressive [tiled]',
    'forward aggressive [tiled] (no fhist)',
]
n = len(fwd_labels)
width = 0.9 / n
fig, ax = plt.subplots(figsize=(14, 5))
for i, label in enumerate(fwd_labels):
    offset = (i - (n - 1) / 2) * width
    ax.bar(x + offset, results[label], width, label=label)
ax.set_xticks(x); ax.set_xticklabels(labels_x)
ax.set_ylabel('Wall time (ms, min of 5)')
ax.set_title('Forward time loop')
ax.legend(fontsize=8, ncol=2, loc='upper left'); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Plot - gradient (forward + adjoint) timing


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
grad_labels = ['grad legacy', 'grad optimized']
width = 0.4
for i, label in enumerate(grad_labels):
    ax.bar(x + (i - 0.5) * width, results[label], width, label=label)
ax.set_xticks(x); ax.set_xticklabels(labels_x)
ax.set_ylabel('Wall time (ms, min of 5)')
ax.set_title('jax.grad through run_forward_jax')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Speedups


In [ ]:
cols = ['opt', 'opt-lite',
        'cpu_f', 'cpu_f-l',
        'gpu_s', 'gpu_s-l',
        'tiled', 'tiled-l',
        'grad opt']
header = f'{"grid":>9} ' + ' '.join(f'{c:>9}' for c in cols)
print(header)
print('-' * len(header))
for k, (nx, nv) in enumerate(grid_sizes):
    base_fwd = results['forward legacy'][k]
    base_grd = results['grad legacy'][k]
    speedups = [
        base_fwd / results['forward optimized'][k],
        base_fwd / results['forward optimized (no fhist)'][k],
        base_fwd / results['forward aggressive [cpu_fused]'][k],
        base_fwd / results['forward aggressive [cpu_fused] (no fhist)'][k],
        base_fwd / results['forward aggressive [gpu_safe]'][k],
        base_fwd / results['forward aggressive [gpu_safe] (no fhist)'][k],
        base_fwd / results['forward aggressive [tiled]'][k],
        base_fwd / results['forward aggressive [tiled] (no fhist)'][k],
        base_grd / results['grad optimized'][k],
    ]
    label = f'{nx}x{nv}'
    print(f'{label:>9} ' + ' '.join(f'{s:8.2f}x' for s in speedups))


## Notes

* On CPU (no CUDA) the optimized path mostly removes Python-side
  bookkeeping and small launch overheads.  Speedups of
  ~5-15% are typical here.
* The aggressive layouts trade differently:
  * `cpu_fused` is fastest on CPU at every grid size we sampled
    (~1.5-1.85x over legacy at 32x32 -> 256x256).  On GPU it
    regresses: the row-sweep collapses parallelism from
    `nx * nv` threads to `nx`, and the cross-thread memory access
    pattern is no longer coalesced.
  * `gpu_safe` keeps `dim=(nx, nv)` parallelism and coalesced
    reads at the cost of one extra rho-reduction kernel per step.
    Should be the safest choice on GPU at large grids.
  * `tiled` is the compromise: F5's memory benefit but launched
    at `dim=(nx, num_tiles)`, with `num_tiles` chosen via
    `aggressive_tile_size`.  `tile_size = warp_size = 32` is a
    reasonable starting point; tune it if you have a profile.
* On a CUDA Warp build the per-step transfer / launch
  elimination across all paths is much more visible, especially
  on the gradient path (legacy vs optimized was 1.6x at 256x256
  on a Colab T4-class GPU).
* `record_history=False` is the cheapest mode whenever you only
  need `f_final`, `E_hist`, or `ee_hist` (all `make_cost_function_*`
  helpers fall in that category).  Available in both optimized
  and aggressive modes.
